# TACO만으로 데이터셋 만들기 (분리수거 2조)

위에서부터 셀을 하나씩 실행(▶)하세요. 전부 돌리면 30분~1시간.

- GPU는 7번(학습 확인)에만 필요: `런타임 → 런타임 유형 변경 → T4 GPU`
- 작업은 Colab 내부 디스크에서 하고, 결과만 압축해서 구글 드라이브에 저장합니다.

## 0. 준비
드라이브에 `분리수거_데이터` 폴더를 만들고 `taco_pipeline.zip`을 넣어 두세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
WORK  = '/content/drive/MyDrive/분리수거_데이터'
LOCAL = '/content/work'
os.makedirs(LOCAL, exist_ok=True)
!unzip -oq "{WORK}/taco_pipeline.zip" -d /content/
!pip -q install pillow requests opencv-python matplotlib
%cd /content/taco_pipeline
!ls

## 1. TACO 받기 + 5클래스 변환
사진 1,500장 다운로드 10~20분. 끝에 `건너뜀: 사진 없음`이 나오면 이 셀을 한 번 더 실행 (받은 건 건너뜀).

In [ ]:
!python s1_taco.py --data /content/taco_data --out {LOCAL}/pool/taco

## 2. train/val 85:15 분할

In [ ]:
!python s7_merge_split.py --pools {LOCAL}/pool/taco --out {LOCAL}/dataset_taco --version taco-v1

## 3. 증강 (train에만)
종이컵이 든 사진은 4장, 유리가 든 사진은 2장, 나머지는 1장씩 증강본을 만듭니다. 배경을 흰색·검정으로 바꾸는 게 핵심.

In [ ]:
!python s8_augment.py --data {LOCAL}/dataset_taco --class-copies 종이컵=4 유리=2

In [ ]:
# 증강본 몇 장 보기 (빨간 선 = 라벨)
import cv2, numpy as np, matplotlib.pyplot as plt
fs = sorted(glob.glob(f'{LOCAL}/dataset_taco/images/train/*_aug1.jpg'))[:6]
plt.figure(figsize=(15, 8))
for k, f in enumerate(fs):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    for line in open(f.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
        p = (np.array(line.split()[1:], float).reshape(-1, 2) * [W, H]).astype(np.int32)
        cv2.polylines(img, [p], True, (255, 0, 0), max(2, W // 300))
    plt.subplot(2, 3, k + 1); plt.imshow(img); plt.axis('off')
plt.show()

## 4. 검수 + 분포 리포트
검수 그림(`check/review/sheet_XX.jpg`)을 보고 `check/review.csv`의 '판정' 칸에 통과/수정/버림을 적어 공유.

In [ ]:
!python s9_check.py --data {LOCAL}/dataset_taco
from IPython.display import Markdown, display, Image as IImage
display(Markdown(open(f'{LOCAL}/dataset_taco/check/report.md', encoding='utf-8').read().split('## 그래프')[0]))
for name in ['split', 'hard_case']:
    p = f'{LOCAL}/dataset_taco/check/charts/{name}.png'
    if os.path.isfile(p): display(IImage(p))
display(IImage(f'{LOCAL}/dataset_taco/check/review/sheet_01.jpg'))

## 5. 드라이브에 저장 (인계)
`dataset_taco.zip`을 팀 공유 폴더에 올리면 모델팀이 `data.yaml`로 바로 학습할 수 있습니다.

In [ ]:
!cd {LOCAL} && zip -qr dataset_taco.zip dataset_taco && cp dataset_taco.zip "{WORK}/"
print('저장 완료:', WORK + '/dataset_taco.zip')
print(open(f'{LOCAL}/dataset_taco/README.md', encoding='utf-8').read()[:1200])

## 6. (선택) 분류 모델용 사진

In [ ]:
!python s10_crops.py --data {LOCAL}/dataset_taco --out {LOCAL}/crops_taco
!cd {LOCAL} && zip -qr crops_taco.zip crops_taco && cp crops_taco.zip "{WORK}/"

## 7. (선택) 학습이 되는지 확인 - GPU 필요
데이터 형식이 맞는지 보는 1 epoch 짧은 학습. 본 학습은 모델팀.

In [ ]:
!pip -q install ultralytics
!yolo segment train data={LOCAL}/dataset_taco/data.yaml model=yolo11n-seg.pt epochs=1 imgsz=640 fraction=0.2 project={LOCAL}/runs name=smoke exist_ok=True